# ID identification
> This part is for identifying ID after your inference

After inference, we will get `aligned_volumes_mip.npy` and `all_neuron_pt_tuple.npy` and an intensity.h5 file, they are all necessary for this part

- `aligned_volumes_mip.npy`: a mip of all volumes, the shape is (y_range, x_range, z_range)
- `all_neuron_pt_tuple.npy`: neuron_pt_tuple of all volumes
- intensity.h5: the intensity we extract

You should download raw image to npy file using `batch_process_folder` from nas to your local machine
And then You should download the `aligned_volumes_mip.npy` and `all_neuron_pt_tuple.npy` in each worm's folder in your local machine and the intensity.h5 in the parent folder    

### download image as npy from nas
- `nas_folder_path`: your nas path containing all your worms
- `save_folder_path`: your local path to save your files
- t_start: npy start volume
- t_end: npy end volume+1

In [ ]:
from utils.read_vols_using_dask import batch_process_folder
nas_folder_path = r"//192.168.1.192/worm-tools/Jinghao-Wang/tea_experiment/20250421_EGCG_high"
save_folder_path = r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG_high"
batch_process_folder(nas_folder_path, save_folder_path, t_start=0, t_end=2)

### download aligned_volumes_mip.npy and all_neuron_pt_tuple.npy from Server
- `REMOTE_BASE_PATH`: your worm folder path containing aligned_volumes_mip.npy and all_neuron_pt_tuple.npy
- `LOCAL_BASE_PATH`: your local path to save your files
- `SERVER_HOSTNAME`: your server hostname
- `SERVER_USERNAME`: your server username
- `SERVER_PASSWORD`: your server password (if not using key-based authentication) None means you already set up key-based authentication
- `SERVER_PORT`: your server SSH port (default is 22)
- `WORM_LIST`: a list of worm names you want to download, None means all worms in the folder

In [ ]:
from id_identify.get_file_from_server import download_files_from_server
SERVER_HOSTNAME = "192.168.1.93"
SERVER_PORT = 23
SERVER_USERNAME = "wangjinghao"
SERVER_PASSWORD = None

REMOTE_BASE_PATH = "/home/data4/WJH/olfactory_result/20240901_wen0065"
LOCAL_BASE_PATH = "I:/WJH/try"

WORM_LIST = None

download_files_from_server(
    hostname=SERVER_HOSTNAME,
    username=SERVER_USERNAME,
    password=SERVER_PASSWORD,
    # key_filename=SSH_PRIVATE_KEY_PATH, # Uncomment if using key-based auth
    port=SERVER_PORT,
    remote_base_path=REMOTE_BASE_PATH,
    local_base_path=LOCAL_BASE_PATH,
    worm_list=WORM_LIST
)

### plot trace use `output_volumes.xlsx` and intensity.h5 file
- `output_volumes.xlsx`: the output file from extract_channel_info.py
- `intensity.h5`: the merged intensity file after inference

- `intensity_path`: the path to the intensity.h5 file
- `labjack_excel_path`: the path to the output_volumes.xlsx file
- `save_folder`: the path to save the trace plot

In [ ]:
from result_plot.draw_signal import draw_raw_signal, draw_trend_signal
intensity_path = r"H:\Process_temporary\WJH\olfactory\ID\result\20250604\20250604.h5"
import h5py
from result_plot.draw_signal import *
with h5py.File(intensity_path, 'r') as f:
    key_list = list(f.keys())

labjack_excel_path = r"H:\Process_temporary\WJH\olfactory\ID\result\20250604\labjack\output_volumes.xlsx"
for key in key_list:
    save_folder = fr"H:\{key}\plot"
    trend_args = {
        "h5_file_path": intensity_path,
        "save_folder": save_folder,
        "exp_name": f"{key}",
        "root_name": key,
        "date": "20250529_odor",
        "labjack_excel_path": labjack_excel_path,
        "n_cols": 2,
        "row_height": 2.5,
        "col_width": 10,
        "xtick_num": 20,
        "alpha": 0.7,
        "ylabel": "deltaF/F_0"
    }
    draw_trend_signal(**trend_args)
    raw_args = {
        "h5_file_path": intensity_path,
        "save_folder": save_folder,
        "exp_name": f"{key}",
        "root_name": key,
        "date": "20250529_odor",
        "labjack_excel_path": labjack_excel_path,
        "n_cols": 2,
        "row_height": 2.5,
        "col_width": 10,
        "xtick_num": 20,
        "alpha": 0.7
    }
    draw_raw_signal(**raw_args)

### use napari to visualize the aligned volumes mip and the neuron points
- `green_file_path`: the path to the green channel npy file you downloaded from nas
- `red_file_path`: the path to the red channel npy file you downloaded from nas
- `aligned_volume_path`: the path to the aligned_volumes_mip.npy file you downloaded from the server
- `neuron_pt_tuple_path`: the path to the all_neuron_pt_tuple.npy
- `mask_name`: a sign name to represent the worm

In [ ]:
from id_identify.id_identify_in_napari import id_identify_in_napari
green_file_path =r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250604_Odor\w1_2025-06-04_15-14-50\green.npy"
red_file_path = r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250604_Odor\w1_2025-06-04_15-14-50\red.npy"
aligned_volume_path = r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250604_Odor\w1_2025-06-04_15-14-50\aligned_volumes_mip.npy"
neuron_pt_tuple_path = r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250604_Odor\w1_2025-06-04_15-14-50\all_neuron_pt_tuple.npy"

viewer = id_identify_in_napari(green_file_path=green_file_path,
                        red_file_path=red_file_path,
                        neuron_pt_tuple_path=neuron_pt_tuple_path,
                        aligned_volume_path=aligned_volume_path,
                        mask_name="w1",
                        green_scale=[1,5,1,1],
                        red_translate=[0,0,-5,1024-5],
                        red_scale=[1, 5, 1, -1],
                        mask_scale=[1, 5, 1, 1])

# Process data and segmentation

## Draw neural signals with bioID
the same function as `draw_neural_signals`, but add one parameter `bi_ID_path` to specify the path to the bioID file

In [ ]:
intensity_path = r"H:\Process_temporary\WJH\olfactory\ID\result\20250604\20250604.h5"
import h5py
from result_plot.draw_signal import *
with h5py.File(intensity_path, 'r') as f:
    key_list = list(f.keys())

labjack_excel_path = r"H:\Process_temporary\WJH\olfactory\ID\result\20250604\output_volumes.xlsx"
for key in key_list:
    save_folder = fr"I:\WJH\flavor\raw_image\20250604\tiffplot\{key}"
    trend_args = {
        "h5_file_path": intensity_path,
        "save_folder": save_folder,
        "exp_name": f"{key}",
        "root_name": key,
        "odor_config_file": r"H:\Process_temporary\WJH\sensory_pipeline_python\data_load\config\compound_info.json",
        "bi_ID_path": r"H:\Process_temporary\WJH\olfactory\ID\result\20250604\ID0604_odor.xlsx",
        "date": "20250604_odor",
        "labjack_excel_path": labjack_excel_path,
        "n_cols": 2,
        "row_height": 2.5,
        "col_width": 10,
        "xtick_num": 20,
        "alpha": 0.7,
        "ylabel": "deltaF/F_0"
    }
    draw_trend_signal(**trend_args)
    raw_args = {
        "h5_file_path": intensity_path,
        "save_folder": save_folder,
        "exp_name": f"{key}",
        "root_name": key,
        "odor_config_file": r"H:\Process_temporary\WJH\sensory_pipeline_python\data_load\config\compound_info.json",
        "bi_ID_path": r"H:\Process_temporary\WJH\olfactory\ID\result\20250604\ID0604_odor.xlsx",
        "date": "20250604_odor",
        "labjack_excel_path": labjack_excel_path,
        "n_cols": 2,
        "row_height": 2.5,
        "col_width": 10,
        "xtick_num": 20,
        "alpha": 0.7
    }
    draw_raw_signal(**raw_args)

## Load worm data from infer result and ID annotation
- get stimulus interval info from output_volumes.xlsx
- get ID data from ID.xlsx
- get intensity data from HDF5 file
- detrend and save data in a dictionary

parameters:
- `h5_file_path`: the path to the intensity.h5 file
- `channel_info_path`: the path to the output_volumes.xlsx file
- `ID_info_path`: the path to the ID.xlsx file
- `date`: the date of the experiment, used to filter the data
- `sorting_config`: a dictionary like follows, maybe useless
```python
    sorting_config_0604 = {
        'w3':{
                'stimulus_sort': [6,7,4,5,2,3,0,1],
                'buffer_sort': [0,7,8,5,6,3,4,1,2]
        },
        'w4':{
                'stimulus_sort': [6,7,4,5,2,3,0,1],
                'buffer_sort': [0,7,8,5,6,3,4,1,2]
        },
        'w6':{
                'stimulus_sort': [6,7,4,5,2,3,0,1],
                'buffer_sort': [0,7,8,5,6,3,4,1,2]
        },
        'w7':{
                'stimulus_sort': [6,7,4,5,2,3,0,1],
                'buffer_sort': [0,7,8,5,6,3,4,1,2]
        },
        'w10':{
                'stimulus_sort': [6,7,4,5,2,3,0,1],
                'buffer_sort': [0,7,8,5,6,3,4,1,2]
        }
    }
```

In [ ]:
from data_load.process_worm_data import load_and_process_worm_data
experiment_df_0604_odor, stimulus_lists_0604_odor, ID_info_0604_odor, worm_data_0604_odor, neuron_segments_dict_0604_odor, neuron_segments_dict_reorganized_0604_odor, neuron_groups_0604_odor = load_and_process_worm_data(
    h5_file_path=  r"H:\Process_temporary\WJH\olfactory\ID\result\20250604\20250604.h5",
    channel_info_path= r"H:\Process_temporary\WJH\olfactory\ID\result\20250604\output_volumes.xlsx",
    ID_info_path= r"H:\Process_temporary\WJH\olfactory\ID\result\20250604\ID0604_odor.xlsx",
    date='20250604',
    sorting_config=None,
)

In [ ]:
from data_load.process_worm_data import merge_multiple_dicts
all_neuron_segments = merge_multiple_dicts(neuron_segments_dict_0604_odor)

import copy
neuron_segments_dict = copy.deepcopy(all_neuron_segments)
# # select distinct keys
# keys_to_copy = ['ASKL', 'ASKR', 'ASJL', 'ASJR', 'ASHL', 'ASHR', 'ADFL', 'ADFR', 'ADLL', 'ADLR', 'ASIL', 'ASIR', 'AWAL', 'AWAR', 'AWBL', 'AWBR', 'AWCL', 'AWCR', 'ASGL', 'ASGR', 'ASER']
# neuron_segments_dict = {key: copy.deepcopy(neuron_segments_dict[key]) for key in keys_to_copy}

## save the segmentation dictionary to a file

In [ ]:
from utils.HDF5Toolkit import save_h5file
save_h5file(r"H:\Process_temporary\WJH\olfactory\ID\result\20250604\neuron_segments_dict.h5", "neuron_segments_dict", mode='w', **neuron_segments_dict)

## Plot with peak response
- `config_file`: a json file contains the compound info
- `output_radar_folder`: the folder to save the output radar plot(absolute peak response)
- `output_heatmap_folder`: the folder to save the output heatmap plot

In [ ]:
import json
from result_plot.radar_heatmap_plot import compare_compounds_and_dilutions, heatmap_plot
import os
config_path = r"H:\Process_temporary\WJH\sensory_pipeline_python\data_load\config\compound_info.json"
output_radar_folder = r"I:\WJH\flavor\odor\radar\pair"
output_heatmap_folder = r"I:\WJH\flavor\odor\Heatmap\pair"

with open(config_path, 'r') as f:
    compounds = json.load(f)

os.makedirs(output_radar_folder, exist_ok=True)
compare_compounds_and_dilutions(neuron_segments_dict, compounds, output_radar_folder, interactive=True,if_combine=False, if_sum_normalization=True)

os.makedirs(output_heatmap_folder, exist_ok=True)
heatmap_plot(neuron_segments_dict, compounds, output_heatmap_folder, if_combine=False)

## plot average and individual trace with dash

In [ ]:
import json
from result_plot.visweb import create_neuronal_dashboard
from utils.HDF5_load import load_h5file
neuron_segments_dict_path = r"H:\Process_temporary\WJH\olfactory\ID\result\20250604\neuron_segments_dict.h5"
config_path = r"H:\Process_temporary\WJH\sensory_pipeline_python\data_load\config\compound_info.json"

neuron_segments_dict = load_h5file(
    neuron_segments_dict_path,
    "neuron_segments_dict"
)

with open(config_path, 'r') as f:
    compounds = json.load(f)

app = create_neuronal_dashboard(neuron_segments_dict, compounds)  
app.run(port=8052, debug=True, jupyter_mode='external')# Change port if needed